# Tandem SAW fluid-flow FNO workflow

This notebook uses the project dataset reader, the existing `FNO3d` training module, and the prediction/plotting module. It predicts `u`, `v`, and `w` (optionally `p`); temperature is never a target.

The training cell expects author-provided fields converted to the format in `flow/tandem_saw/README.md`. It does not generate CFD data. Update the data path and validation case IDs when those files are available.

In [ ]:
from pathlib import Path
import sys
import json
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Locate the repository root whether Jupyter starts here or at the project root.
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'core' / 'fno.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'core' / 'fno.py').exists():
    raise RuntimeError('Could not locate Navier project root from the notebook working directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from flow.tandem_saw.dataset import TandemSAWDataset
from flow.tandem_saw.training import train_model
from flow.tandem_saw.visualize import predict_sample, plot_prediction, plot_training_history

DATA_PATH = PROJECT_ROOT / 'flow' / 'tandem_saw' / 'data' / 'author_fields.pt'
RUNS_DIR = PROJECT_ROOT / 'flow' / 'tandem_saw' / 'runs'
VALIDATION_CASES = ['Case4']  # Change to IDs present in the author dataset.
RUN_TRAINING = True
print('Project:', PROJECT_ROOT)
print('Author data:', DATA_PATH)

In [ ]:
if DATA_PATH.exists():
    dataset = TandemSAWDataset(DATA_PATH)
    print('Samples:', len(dataset))
    print('X shape:', tuple(dataset.X.shape))
    print('Y shape:', tuple(dataset.Y.shape))
    print('Cases:', sorted(set(dataset.case_id)))
    print('Input fields:', dataset.input_channels)
    print('Flow targets:', dataset.output_channels)
else:
    dataset = None
    display(Markdown('**Author dataset not found.** Add the authors’ exported fields at `flow/tandem_saw/data/author_fields.pt` after converting them to the documented tensor format. No synthetic data is created by this notebook.'))

## Train and validate

Each invocation creates a timestamped run folder containing `run.log`, `config.json`, `history.jsonl`, `summary.json`, and the best checkpoint. The split is by complete case, so adjacent snapshots from one welding case cannot appear in both training and validation.

In [ ]:
run_result = None
if dataset is not None and RUN_TRAINING:
    available_cases = set(dataset.case_id)
    missing_cases = set(VALIDATION_CASES) - available_cases
    if missing_cases:
        raise ValueError(f'Update VALIDATION_CASES; not found: {sorted(missing_cases)}. Available: {sorted(available_cases)}')
    run_result = train_model(
        data_path=DATA_PATH,
        val_cases=VALIDATION_CASES,
        runs_dir=RUNS_DIR,
        epochs=100,
        batch_size=2,
        modes=(16, 16, 12),
        width=32,
        layers=8,
        learning_rate=1e-3,
        weight_decay=1e-4,
        seed=42,
    )
    print('Run folder:', run_result['run_dir'])
    print('Checkpoint:', run_result['checkpoint_path'])
elif dataset is not None:
    print('Training skipped. Set RUN_TRAINING = True to start a new run.')

In [ ]:
if run_result is not None:
    fig, axes = plot_training_history(run_result['history'])
    display(fig)
    plt.close(fig)
    display(Markdown('### Best validation result'))
    display(run_result['summary']['best_validation_metrics'])
else:
    print('Training history will appear here after a run completes.')

## Compare a validation sample

Choose a sample belonging to one of the held-out cases. The figure displays the author target, FNO prediction, and signed error on the three central coordinate planes. Run this once a checkpoint exists.

In [ ]:
if dataset is not None and run_result is not None:
    checkpoint_path = run_result['checkpoint_path']
    validation_indices = [i for i, case in enumerate(dataset.case_id) if case in set(VALIDATION_CASES)]
    sample_index = validation_indices[0]
    field_to_plot = 'u'
    result = predict_sample(dataset, checkpoint_path, sample_index)
    fig, axes = plot_prediction(result, field=field_to_plot)
    display(fig)
    plt.close(fig)
else:
    print('A prepared author dataset and completed run are required before plotting predictions.')

In [ ]:
# Optional: inspect the detailed run log and saved configuration.
if run_result is not None:
    print((run_result['run_dir'] / 'run.log').read_text())
    display(json.loads((run_result['run_dir'] / 'config.json').read_text()))